# CS570 Team 14 — GCN Modifiability Prediction
## Colab Setup & Training Notebook

---

## ⚠️ BEFORE YOU RUN ANYTHING — Read This First

There are a few things you need to do manually before the code will work.
Do these **once** at the start of every new Colab session.

---

### Manual Step 1 — Enable GPU

By default Colab gives you a CPU, which is too slow for training. You need to switch to GPU:

1. In the top menu, click **Runtime**
2. Click **Change runtime type**
3. Under **Hardware accelerator**, select **T4 GPU**
4. Click **Save**
5. Colab will restart — that's normal

How to verify it worked: after the restart, the top-right of the page should show a green chip that says **T4**. If it says **CPU**, repeat the steps above.

---

### Manual Step 2 — Google Account

You need to be logged into a Google account. Colab will ask you to authorize access to Google Drive in Section 2 of the notebook. When the pop-up appears:

1. Click the link it gives you
2. Choose your Google account
3. Click **Allow**
4. Copy the authorization code and paste it back into the Colab input box
5. Press Enter

You will see: `Mounted at /content/drive` — that means it worked.

---

### Manual Step 3 — Rico Dataset (one-time only)

**Your laptop is not involved at all.** The notebook downloads the Rico dataset directly from the internet into your Google Drive using Colab's own internet connection, which is much faster.

The only thing that can go wrong is if the download URL is broken. In that case:
1. Ask your instructor or a labmate if they have a Google Drive share link for the Rico dataset
2. If yes, run this in a Colab cell (replace `FILE_ID` with the ID from their link):
   ```
   !gdown "https://drive.google.com/uc?id=FILE_ID" -O /content/drive/MyDrive/cs570-project/ui_layout_vectors.zip
   ```
3. If no share link is available either, as a last resort download to your laptop from http://interactionmining.org/rico and upload to `My Drive > cs570-project`

You only need to do this **once ever**. After the `.pt` processed files are on Drive, you never need the raw JSONs again and this whole section is skipped automatically.

---

### What lives where

| Data | Location | Persists after session? |
|------|----------|--------------------------|
| Rico zip (6 GB) | Google Drive | ✅ Yes — downloaded once, stays there |
| Rico raw JSONs (unzipped) | `/content/rico_raw/` (local) | ❌ No — unzipped fresh each time from Drive zip, only needed for preprocessing |
| Processed `.pt` graphs | Google Drive | ✅ Yes — the main output, reused every session |
| Model checkpoints | Google Drive | ✅ Yes |
| Embedding cache | Google Drive | ✅ Yes |
| This repo | `/content/cs570-aiml/` (local) | ❌ No — re-cloned each session in ~30 seconds |

**The key insight:** preprocessing only runs once. After that, every new session skips straight to training by loading `.pt` files from Drive.

---

### How to run the notebook

Run cells **top to bottom, one at a time**. Each cell has a ▶ button on the left. Wait for one cell to finish (the `[*]` next to it becomes a number like `[3]`) before running the next one.

If a cell throws an error, read the last line of the error — it usually tells you exactly what's wrong. The most common issues are covered in the **Troubleshooting** section at the bottom.

---
## Section 1 — Check GPU

Run this first to confirm you have a GPU. If it says `Failed to initialize NVML` or shows no GPU, go back and do Manual Step 1.

In [32]:
!nvidia-smi

Tue May 26 05:24:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             16W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [33]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)} GB")
else:
    print("❌ No GPU found. Go to Runtime > Change runtime type > T4 GPU, then re-run.")

✅ GPU ready: Tesla T4
   VRAM: 15.6 GB


---
## Section 2 — Mount Google Drive

This will pop up an authorization prompt. Follow Manual Step 2 above.

In [34]:
from google.colab import drive
drive.mount('/content/drive')

import os
if os.path.exists('/content/drive/MyDrive'):
    print("✅ Drive mounted successfully")
else:
    print("❌ Drive mount failed — try running this cell again")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Drive mounted successfully


---
## Section 3 — Clone Repo from GitHub

In [ ]:
import os

REPO_URL = "https://github.com/nadiarvi/cs570-aiml"
REPO_DIR = "/content/cs570-aiml"
BRANCH   = "colab"

if os.path.exists(REPO_DIR):
    print("Repo already cloned — pulling latest changes...")
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    print("Cloning repo...")
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"\n✅ Working directory: {os.getcwd()}")

In [36]:
import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
print("✅ Repo added to Python path")

✅ Repo added to Python path


---
## Section 4 — Install Dependencies

Torch, numpy, and pandas are pre-installed in Colab. This just installs the missing ones.
Takes about 1–2 minutes.

In [37]:
!pip install -q sentence-transformers scikit-learn seaborn tqdm
print("✅ Dependencies installed")

✅ Dependencies installed


---
## Section 5 — Set Up Paths

- **Drive** (`DRIVE_ROOT`): stores processed graphs, checkpoints, embedding cache — persists between sessions
- **Local** (`RICO_RAW_DIR`): stores raw Rico JSONs temporarily — fast to work with, gets wiped when session ends (that's fine)

In [38]:
import os

# ── Persistent (Google Drive) ──────────────────────────────────────────────
DRIVE_ROOT      = "/content/drive/MyDrive/cs570-project"
PROCESSED_DIR   = f"{DRIVE_ROOT}/data/processed"
SPLITS_DIR      = f"{DRIVE_ROOT}/data/splits"
GOLD_DIR        = f"{DRIVE_ROOT}/data/gold"
CHECKPOINTS     = f"{DRIVE_ROOT}/results/checkpoints"
RESULTS_DIR     = f"{DRIVE_ROOT}/results"
EMBEDDING_CACHE = f"{DRIVE_ROOT}/embedding_cache.json"

RICO_TAR_ON_DRIVE = f"{DRIVE_ROOT}/unique_uis.tar.gz"  # zip lives here permanently

# ── Ephemeral (local Colab storage — fast, wiped on session end) ───────────
RICO_RAW_DIR  = "/content/rico_raw"   # raw JSONs live here during preprocessing
RICO_TAR_LOCAL = "/content/unique_uis.tar.gz"

# Create all Drive folders
for path in [PROCESSED_DIR, SPLITS_DIR, GOLD_DIR, CHECKPOINTS,
             f"{RESULTS_DIR}/logs", f"{RESULTS_DIR}/figures"]:
    os.makedirs(path, exist_ok=True)

os.makedirs(RICO_RAW_DIR, exist_ok=True)

print("Drive folders:")
print(f"  processed graphs → {PROCESSED_DIR}")
print(f"  checkpoints      → {CHECKPOINTS}")
print(f"  embedding cache  → {EMBEDDING_CACHE}")
print(f"\nLocal (temporary):")
print(f"  rico raw JSONs   → {RICO_RAW_DIR}")
print("\n✅ Paths configured")

Drive folders:
  processed graphs → /content/drive/MyDrive/cs570-project/data/processed
  checkpoints      → /content/drive/MyDrive/cs570-project/results/checkpoints
  embedding cache  → /content/drive/MyDrive/cs570-project/embedding_cache.json

Local (temporary):
  rico raw JSONs   → /content/rico_raw

✅ Paths configured


---
## Section 6 — Get Rico Dataset

**Skip this entire section if you have already preprocessed the data in a previous session.**
Run the check cell first to see if you need to re-download.

In [39]:
# Check if processed graphs already exist on Drive (from a previous session)
import glob

existing_pt = glob.glob(f"{PROCESSED_DIR}/**/*.pt", recursive=True)
if existing_pt:
    print(f"✅ Found {len(existing_pt)} processed .pt files on Drive.")
    print("   You can skip Section 6 and go straight to Section 7 (training).")
else:
    print("⚠️  No processed graphs found. You need to download + preprocess Rico.")
    print("   Continue running the cells below.")

⚠️  No processed graphs found. You need to download + preprocess Rico.
   Continue running the cells below.


In [40]:
# ── Download Rico directly to Google Drive ────────────────────────────────
# Downloads from Google Cloud Storage straight into Drive.
# Your laptop is not involved. Takes ~2–5 minutes.
# The tar.gz stays on Drive permanently — never re-downloaded.

RICO_URL = "https://storage.googleapis.com/crowdstf-rico-uiuc-4540/rico_dataset_v0.1/unique_uis.tar.gz"

if os.path.exists(RICO_TAR_ON_DRIVE) and os.path.getsize(RICO_TAR_ON_DRIVE) > 1e8:
    print(f"✅ Rico archive already on Drive ({os.path.getsize(RICO_TAR_ON_DRIVE)/1e9:.1f} GB) — skipping download")
else:
    print("Downloading Rico dataset (~6 GB) directly to Google Drive...")
    print("This takes 2–5 minutes. Do not close the tab.")
    !wget -q --show-progress "{RICO_URL}" -O "{RICO_TAR_ON_DRIVE}"

    if os.path.exists(RICO_TAR_ON_DRIVE) and os.path.getsize(RICO_TAR_ON_DRIVE) > 1e8:
        print(f"✅ Download complete — saved to Drive ({os.path.getsize(RICO_TAR_ON_DRIVE)/1e9:.1f} GB)")
    else:
        print("❌ Download failed or file too small. Run the fallback cell below.")

This takes 2–5 minutes. Do not close the tab.
/content/drive/MyDr 100%[===================>]   6.03G  46.2MB/s    in 2m 19s  
✅ Download complete — saved to Drive (6.5 GB)


In [41]:
# ── Fallback: manual upload ────────────────────────────────────────────────
# ONLY run this if the download cell above failed.
#
# What to do:
#   1. On your laptop, go to this URL and download the file:
#      https://storage.googleapis.com/crowdstf-rico-uiuc-4540/rico_dataset_v0.1/unique_uis.tar.gz
#   2. Go to https://drive.google.com
#   3. Navigate to: My Drive > cs570-project
#   4. Upload unique_uis.tar.gz into that folder
#   5. Wait for the upload to finish (watch the progress bar in Drive bottom-right)
#   6. Then run this cell to verify

if os.path.exists(RICO_TAR_ON_DRIVE) and os.path.getsize(RICO_TAR_ON_DRIVE) > 1e8:
    print(f"✅ Found archive on Drive ({os.path.getsize(RICO_TAR_ON_DRIVE)/1e9:.1f} GB) — ready to extract")
else:
    print(f"❌ Not found at: {RICO_TAR_ON_DRIVE}")
    print("   Make sure the upload finished and the filename is exactly: unique_uis.tar.gz")

✅ Found archive on Drive (6.5 GB) — ready to extract


In [42]:
# ── Extract archive to local storage ──────────────��───────────────────────
# Copies the tar.gz from Drive to local /content/, then extracts there.
# Extracting locally is much faster than extracting directly on Drive.
# The raw JSONs in /content/rico_raw/ are ephemeral — that's fine because
# preprocessing only needs to run once and saves .pt files to Drive.

import glob as glob_module

existing_jsons = glob_module.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)

if len(existing_jsons) > 1000:
    print(f"✅ Already extracted locally ({len(existing_jsons)} JSON files) — skipping")

elif not os.path.exists(RICO_TAR_ON_DRIVE) or os.path.getsize(RICO_TAR_ON_DRIVE) < 1e8:
    print("❌ Archive not found on Drive. Run the download cell first.")

else:
    print(f"Copying archive from Drive to local storage (~1–2 min)...")
    !cp "{RICO_TAR_ON_DRIVE}" "{RICO_TAR_LOCAL}"
    print("✅ Copied")

    print(f"\nExtracting → {RICO_RAW_DIR}  (~5–10 minutes)...")
    !tar -xzf "{RICO_TAR_LOCAL}" -C "{RICO_RAW_DIR}"

    # Delete local copy of archive to free space (still safe on Drive)
    os.remove(RICO_TAR_LOCAL)

    extracted = glob_module.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)
    print(f"✅ Extracted {len(extracted)} JSON files")

Copying archive from Drive to local storage (~1–2 min)...
✅ Copied

Extracting → /content/rico_raw  (~5–10 minutes)...
✅ Extracted 66261 JSON files


In [43]:
# Verify extraction and find where the JSONs actually landed
import glob as glob_module

json_files = glob_module.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)
print(f"Rico JSON files found: {len(json_files)}")

if len(json_files) > 10000:
    # Show the directory structure so we know the exact path to pass to preprocessing
    sample = json_files[0]
    print(f"✅ Looks good — ready to preprocess")
    print(f"   Sample path: {sample}")
    # The rico_dir for preprocessing should be the folder containing app_id subfolders
    # Typically: RICO_RAW_DIR/combined/ or just RICO_RAW_DIR/
    parts = sample.replace(RICO_RAW_DIR + "/", "").split("/")
    if len(parts) >= 3:
        RICO_JSON_DIR = f"{RICO_RAW_DIR}/{parts[0]}"
        print(f"   Use this as --rico_dir: {RICO_JSON_DIR}")
    else:
        RICO_JSON_DIR = RICO_RAW_DIR
        print(f"   Use this as --rico_dir: {RICO_JSON_DIR}")
elif len(json_files) > 0:
    print(f"⚠️  Only {len(json_files)} files. Expected ~66,000. Extraction may be incomplete.")
    RICO_JSON_DIR = RICO_RAW_DIR
else:
    print("❌ No JSON files found. Check the extraction step.")
    RICO_JSON_DIR = RICO_RAW_DIR

print(f"\nRICO_JSON_DIR = '{RICO_JSON_DIR}'")

Rico JSON files found: 66261
✅ Looks good — ready to preprocess
   Sample path: /content/rico_raw/combined/6661.json
   Use this as --rico_dir: /content/rico_raw

RICO_JSON_DIR = '/content/rico_raw'


---
## Section 7 — Preprocess

Converts raw Rico JSONs → processed graph `.pt` files saved to Drive.

**This is the slowest step** because it runs MiniLM text embeddings on every node's text.
The embedding cache (`embedding_cache.json`) saves results so repeated runs are fast.

**Strategy:**
1. Run the 500-screen sanity check first (~3 minutes) — confirms the pipeline works
2. Then run 10,000 screens (~45–90 minutes) — enough to train a good model
3. Full 66K only if you need it later

**Skip this section** if you already have `.pt` files on Drive from a previous session.

In [44]:
# ── Step 7a: Sanity check — 500 screens ───────��───────────────────────────
# Run this first to confirm the pipeline works before the full run.

!python src/data/preprocess.py \
    --rico_dir             {RICO_JSON_DIR} \
    --out_dir              {PROCESSED_DIR} \
    --split_path           {SPLITS_DIR}/split_seed42.json \
    --label_mode           contextual \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers              1 \
    --max_screens          500

Traceback (most recent call last):
  File "/content/cs570-aiml/src/data/preprocess.py", line 23, in <module>
    from src.data.rico_loader import load_hierarchy, flatten_hierarchy, get_app_id
ModuleNotFoundError: No module named 'src'


In [ ]:
import glob, torch

train_pts = glob.glob(f"{PROCESSED_DIR}/train/**/*.pt", recursive=True)
val_pts   = glob.glob(f"{PROCESSED_DIR}/val/**/*.pt",   recursive=True)
other_pts = glob.glob(f"{PROCESSED_DIR}/other/**/*.pt", recursive=True)

print(f"Train graphs : {len(train_pts)}")
print(f"Val graphs   : {len(val_pts)}")
print(f"Other (bad)  : {len(other_pts)}  ← should be 0; if not, re-run preprocessing")

if train_pts:
    sample = torch.load(train_pts[0], weights_only=False)
    print(f"\nSample graph: {train_pts[0]}")
    print(f"  Nodes       : {sample['num_nodes']}")
    print(f"  Feature dim : {sample['x'].shape[1]}")
    print(f"  Edges       : {sample['edge_index'].shape[1]}")
    print(f"  Labels      : { {int(k): int(v) for k, v in zip(*sample['y'].unique(return_counts=True))} }")
    print("\n✅ Ready to train")
elif other_pts:
    print("\n❌ Files went to 'other/' — re-run Section 3 (git pull) then re-run this preprocessing cell")
else:
    print("\n❌ No .pt files found at all — run preprocessing first")

In [46]:
# ── Step 7b: Full preprocessing — 10K screens, contextual labels ──────────
# Already-processed files are skipped, so re-running is safe.
# Expect ~45–90 minutes.

!python src/data/preprocess.py \
    --rico_dir             {RICO_JSON_DIR} \
    --out_dir              {PROCESSED_DIR} \
    --split_path           {SPLITS_DIR}/split_seed42.json \
    --label_mode           contextual \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers              1 \
    --max_screens          10000

Traceback (most recent call last):
  File "/content/cs570-aiml/src/data/preprocess.py", line 23, in <module>
    from src.data.rico_loader import load_hierarchy, flatten_hierarchy, get_app_id
ModuleNotFoundError: No module named 'src'


In [47]:
# ── Step 7c: Full preprocessing — local_only labels ───────────────────────
# Needed for the circularity safeguard experiment. Run after 7b finishes.

!python src/data/preprocess.py \
    --rico_dir             {RICO_JSON_DIR} \
    --out_dir              {PROCESSED_DIR} \
    --split_path           {SPLITS_DIR}/split_seed42.json \
    --label_mode           local_only \
    --embedding_cache_path {EMBEDDING_CACHE} \
    --workers              1 \
    --max_screens          10000

Traceback (most recent call last):
  File "/content/cs570-aiml/src/data/preprocess.py", line 23, in <module>
    from src.data.rico_loader import load_hierarchy, flatten_hierarchy, get_app_id
ModuleNotFoundError: No module named 'src'


In [48]:
# Final verification before training
for mode in ["contextual", "local_only"]:
    train_pts = glob.glob(f"{PROCESSED_DIR}/train/{mode}/**/*.pt", recursive=True)
    val_pts   = glob.glob(f"{PROCESSED_DIR}/val/{mode}/**/*.pt",   recursive=True)
    print(f"{mode:12s}  train={len(train_pts):5d}  val={len(val_pts):5d}")

contextual    train=    0  val=    0
local_only    train=    0  val=    0


---
## Section 8 — Train a Single Model

Trains GCN on heuristic contextual labels. Early-stops on heuristic val Macro-F1.
Checkpoint is saved to Drive so you can resume in a future session.

With 10K screens and 100 epochs, expect **30–60 minutes**.

In [ ]:
# ── Auto-preprocess if needed ──────────────────────────────────────────────
# Checks whether processed .pt files exist on Drive. If not, automatically
# detects the Rico JSON location and runs preprocessing before training.

import glob as _glob, os as _os

DRIVE_ROOT    = "/content/drive/MyDrive/cs570-project"
PROCESSED_DIR = f"{DRIVE_ROOT}/data/processed"
SPLITS_DIR    = f"{DRIVE_ROOT}/data/splits"
EMBEDDING_CACHE = f"{DRIVE_ROOT}/embedding_cache.json"
RICO_RAW_DIR  = "/content/rico_raw"

train_pts = _glob.glob(f"{PROCESSED_DIR}/train/**/*.pt", recursive=True)

if train_pts:
    print(f"✅ {len(train_pts)} train graphs already on Drive — skipping preprocessing")
else:
    print("No processed graphs found — running preprocessing now...\n")

    # Auto-detect where the JSONs are (handles subfolder like combined/)
    json_files = _glob.glob(f"{RICO_RAW_DIR}/**/*.json", recursive=True)
    if not json_files:
        raise FileNotFoundError(
            "Rico JSON files not found in /content/rico_raw/\n"
            "The Rico data needs to be extracted first.\n"
            "Re-run Section 6 (download + extract) then come back here."
        )

    sample = json_files[0].replace(RICO_RAW_DIR + "/", "")
    parts  = sample.split("/")
    RICO_JSON_DIR = f"{RICO_RAW_DIR}/{parts[0]}" if len(parts) >= 3 else RICO_RAW_DIR
    print(f"Rico JSONs found: {len(json_files)} files")
    print(f"Using rico_dir : {RICO_JSON_DIR}\n")

    for label_mode in ["contextual", "local_only"]:
        print(f"--- Preprocessing: {label_mode} ---")
        !python /content/cs570-aiml/src/data/preprocess.py \
            --rico_dir             {RICO_JSON_DIR} \
            --out_dir              {PROCESSED_DIR} \
            --split_path           {SPLITS_DIR}/split_seed42.json \
            --label_mode           {label_mode} \
            --embedding_cache_path {EMBEDDING_CACHE} \
            --workers              1 \
            --max_screens          10000

    train_pts = _glob.glob(f"{PROCESSED_DIR}/train/**/*.pt", recursive=True)
    val_pts   = _glob.glob(f"{PROCESSED_DIR}/val/**/*.pt",   recursive=True)
    print(f"\n✅ Preprocessing done — train={len(train_pts)}  val={len(val_pts)}")

In [49]:
import json
import logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")

from src.train import train

with open(f"{REPO_DIR}/experiments/configs/colab_gcn_2l_all_contextual.json") as f:
    config = json.load(f)

# Override with actual runtime paths
config["processed_dir"] = PROCESSED_DIR
config["save_dir"]       = f"{CHECKPOINTS}/gcn_2l_all_contextual"
config["save_root"]      = CHECKPOINTS

metadata = train(config)
print(f"\n✅ Training complete")
print(f"   Best val Macro-F1 : {metadata['best_val_macro_f1']:.4f}")
print(f"   Best epoch         : {metadata['best_epoch']}")
print(f"   Checkpoint saved to: {metadata['checkpoint_path']}")

FileNotFoundError: No training graphs found in /content/drive/MyDrive/cs570-project/data/processed/train/contextual/**/*.pt

---
## Section 9 — Gold Labels

⚠️ **Manual step required before running this section.**

You need to create `gold_test_labels.csv` and upload it to Drive before evaluation.
This file contains the hand-labeled annotations for ~50 Rico screens.

**What to do:**
1. Annotate ~50 screens from apps that are NOT in the heuristic train/val split
   - To find which apps are in the gold holdout, look at `data/splits/split_seed42.json` on Drive after preprocessing — specifically the `gold_app_ids` list
   - If you haven't decided which apps to annotate yet, pick any 50 apps NOT in `train_app_ids` or `val_app_ids`
2. Create a CSV file with these columns: `screen_id, node_id, label, annotator_id, notes`
   - `label` must be one of: `canonical`, `translatable`, `open`
   - `node_id` must match the `node_id` field in the flattened hierarchy (run the inspect cell below to check)
3. Save the file as `gold_test_labels.csv`
4. Upload it to `My Drive > cs570-project > data > gold` in Google Drive
5. Then run the evaluation cells below

**The cell below lets you inspect node IDs for a specific screen to help with annotation.**

In [ ]:
# Inspect node IDs for a specific screen to help with annotation
# Change SCREEN_PATH to the .pt file you want to annotate

import torch, json

# List a few available screens to pick from
sample_pts = glob.glob(f"{PROCESSED_DIR}/train/contextual/**/*.pt", recursive=True)[:5]
print("Sample .pt files available:")
for p in sample_pts:
    print(" ", p)

print("\nChange SCREEN_PATH below to inspect a specific screen's nodes:")
SCREEN_PATH = sample_pts[0] if sample_pts else None

if SCREEN_PATH:
    g = torch.load(SCREEN_PATH, weights_only=False)
    print(f"\nScreen: {g['screen_id']}  App: {g['app_id']}  Nodes: {g['num_nodes']}")
    print(f"{'node_id':>8}  {'label':>12}  {'depth':>5}")
    print("-" * 30)
    label_names = {0: 'canonical', 1: 'translatable', 2: 'open', -1: 'excluded'}
    for i, (nid, lbl) in enumerate(zip(g['node_ids'], g['y'].tolist())):
        print(f"{nid:>8}  {label_names.get(lbl, str(lbl)):>12}")
        if i >= 20:
            print(f"  ... ({g['num_nodes'] - 21} more nodes)")
            break

---
## Section 10 — Run Full Ablation Matrix

Trains all 9 experiment configs and evaluates on gold labels.
Results are saved to `ablation_results.csv` on Drive.

**Requires:** gold labels uploaded to Drive (Section 9).

This will take several hours — one run per config × 9 configs.

In [ ]:
from src.ablation import run_ablation

with open(f"{REPO_DIR}/experiments/configs/colab_ablation_base.json") as f:
    base_config = json.load(f)

base_config["processed_dir"]    = PROCESSED_DIR
base_config["gold_labels_path"] = f"{GOLD_DIR}/gold_test_labels.csv"
base_config["save_root"]        = CHECKPOINTS

out_csv = f"{RESULTS_DIR}/ablation_results.csv"
run_ablation(base_config, out_csv=out_csv)
print(f"\n✅ Ablation complete — results at {out_csv}")

In [ ]:
# View results table
import pandas as pd

results = pd.read_csv(out_csv)
cols = ["name", "model", "label_mode", "heuristic_val_macro_f1", "gold_macro_f1", "gold_accuracy"]
print(results[cols].sort_values("gold_macro_f1", ascending=False).to_string(index=False))

---
## Section 11 — Push Results to GitHub

In [ ]:
# Copy results from Drive into the repo, then push
!cp {out_csv} {REPO_DIR}/results/ablation_results.csv

!git -C {REPO_DIR} config user.email "nadia.arvi@gmail.com"
!git -C {REPO_DIR} config user.name "nadiarvi"
!git -C {REPO_DIR} add results/ablation_results.csv
!git -C {REPO_DIR} commit -m "Add ablation results from Colab run"
!git -C {REPO_DIR} push origin colab

---
## Utilities

In [ ]:
# Check GPU memory usage (run this while training is happening in another cell)
!nvidia-smi --query-gpu=memory.used,memory.free,utilization.gpu --format=csv

In [ ]:
# Check how much Drive space you're using
!du -sh {DRIVE_ROOT}/*

In [ ]:
# Check how much time is left in your Colab session
# (Colab free tier disconnects after ~12 hours, Pro after ~24 hours)
!cat /proc/uptime | awk '{print "Session uptime:", int($1/3600), "hours", int(($1%3600)/60), "minutes"}'

---
## Troubleshooting

**`FileNotFoundError: No training graphs found in .../processed/train/contextual/**/*.pt`**
> Preprocessing hasn't run yet. Go to Section 7 and run those cells first.

**`ModuleNotFoundError: No module named 'src'`**
> The repo isn't on the Python path. Re-run the sys.path cell in Section 3.

**`ModuleNotFoundError: No module named 'sentence_transformers'`**
> Re-run Section 4 (pip install cell).

**`RuntimeError: CUDA out of memory`**
> Reduce `batch_size` in the config (try 16 or 8), then re-run the training cell.

**Drive mount shows an error or hangs**
> Run `drive.flush_and_unmount()` then re-run the mount cell. If it still hangs, go to Runtime > Restart runtime and start from Section 1.

**Colab session disconnected mid-training**
> The checkpoint is saved to Drive after every improvement. Re-run Sections 1–4 (setup), then re-run the training cell — it will resume from where it left off if you load the checkpoint. Currently training restarts from scratch; if this is a problem let me know and I can add checkpoint resuming.

**`wget` download is very slow or fails**
> The Rico server can be unreliable. Try the manual upload fallback in Section 6.

**Preprocessing is stuck on the same screen for a long time**
> The MiniLM model is being downloaded on the first run (~90 MB). It'll be fast after that.

**`git push` asks for a password**
> GitHub no longer accepts passwords. Use a Personal Access Token instead:
> 1. Go to https://github.com/settings/tokens
> 2. Click `Generate new token (classic)`
> 3. Check the `repo` scope
> 4. Copy the token
> 5. When git asks for password, paste the token instead